# Static Typing & Linting Tools: `mypy` vs `pyright` vs `ruff`

These three tools are often mentioned together but do **different jobs**. The
single most common confusion:

> **`mypy` and `pyright` are *type checkers*. `ruff` is a *linter + formatter*,
> not a type checker.** They are complements, not competitors.

Companion to [`typed_contracts.ipynb`](typed_contracts.ipynb).

## 1. What each tool actually does

| | **mypy** | **pyright** | **ruff** |
| --- | --- | --- | --- |
| Category | Static **type checker** | Static **type checker** | **Linter + formatter** |
| Made by | Python community (Dropbox-origin) | Microsoft | Astral |
| Written in | Python (+ mypyc) | TypeScript/Node | Rust |
| Speed | Moderate | Fast | **Extremely fast** (10–100×) |
| Checks types? | ✅ yes | ✅ yes | ❌ no (lint rules only) |
| Lints style/bugs? | ❌ | ❌ | ✅ (flake8/pylint-style rules) |
| Formats code? | ❌ | ❌ | ✅ (Black-compatible) |
| Editor backing | via plugins | **Pylance** in VS Code | ruff LSP / many editors |
| Config | `mypy.ini` / `pyproject.toml` | `pyrightconfig.json` / `pyproject.toml` | `pyproject.toml` / `ruff.toml` |

The takeaway: **pick one type checker (mypy or pyright)** and **use ruff** for
everything a type checker doesn't do (imports, unused vars, style, formatting,
upgrades).


## 2. `ruff` — linter + formatter (the "fast everything-else")

Ruff replaces a whole stack of older tools with one Rust binary:
`flake8` (+ dozens of plugins), `isort`, `pyupgrade`, `pydocstyle`, `pylint`
(subset), and **`black`** (formatter). It autofixes most issues.

```bash
pip install ruff

ruff check .            # lint
ruff check . --fix      # lint + autofix
ruff format .           # format (Black-compatible)
```

Rule families are referenced by code prefix and enabled in config:

```toml
# pyproject.toml
[tool.ruff]
line-length = 88
target-version = "py312"

[tool.ruff.lint]
# E/W=pycodestyle, F=pyflakes, I=isort, UP=pyupgrade, B=bugbear, SIM=simplify
select = ["E", "F", "W", "I", "UP", "B", "SIM"]
ignore = ["E501"]       # let the formatter handle line length
```

> **Ruff does not do type checking.** It can flag *some* annotation **style**
> issues, but it never infers or verifies types. (Astral's separate type
> checker, **`ty`**, is in preview as of 2025–2026 — still pair Ruff with a real
> type checker today.)


## 3. `mypy` — the reference type checker

```bash
pip install mypy
mypy your_package/
```

Characteristics:
* **Gradual typing**: unannotated code is treated as `Any` and largely skipped —
  adopt types incrementally. Tighten with `--strict`.
* **Reference semantics**: mypy is maintained alongside the typing PEPs, so its
  behavior is often the de-facto standard.
* **Plugin ecosystem**: e.g. `pydantic.mypy`, `django-stubs`, `sqlalchemy`
  plugins teach mypy framework-specific magic.
* **Pace**: slower than pyright; caches between runs (`.mypy_cache`).

```toml
# pyproject.toml
[tool.mypy]
python_version = "3.12"
strict = true                 # turn on the full strict bundle
warn_unused_ignores = true
plugins = ["pydantic.mypy"]

[[tool.mypy.overrides]]
module = "untyped_dep.*"
ignore_missing_imports = true
```


## 4. `pyright` — fast checker + the engine behind Pylance

```bash
pip install pyright       # ships a Node-based CLI
pyright                    # check the project
pyright --watch           # re-check on save
```

Characteristics:
* **Speed**: typically much faster than mypy on large codebases; great for
  watch mode and editor feedback.
* **Stronger inference**: more aggressive flow-based narrowing, so it often
  catches issues mypy misses (and occasionally reports ones mypy won't).
* **Pylance**: pyright is the type engine inside VS Code's Pylance extension —
  if you use VS Code you are *already running pyright* as you type.
* **Two baselines**: `basic` (lenient) and `strict` (everything on), per-file or
  per-directory.

```json
// pyrightconfig.json
{
  "include": ["src"],
  "pythonVersion": "3.12",
  "typeCheckingMode": "strict",
  "reportMissingTypeStubs": false
}
```
(Equivalent settings can also live under `[tool.pyright]` in `pyproject.toml`.)


## 5. `mypy` vs `pyright` — choosing a type checker

| Dimension | mypy | pyright |
| --------- | ---- | ------- |
| Speed | moderate | **fast** (watch-mode friendly) |
| Type inference | solid, conservative | **more aggressive** narrowing |
| Editor integration | via LSP plugins | **native in VS Code** (Pylance) |
| Plugins / framework magic | **yes** (pydantic, django, sqlalchemy) | no plugin system (relies on stubs) |
| Install footprint | pure Python | bundles Node |
| "Reference" behavior | **closest to the PEPs** | very standards-compliant, some extras |
| Strictness control | flags + `--strict` | `basic`/`strict` modes, per-path |

They will **not always agree** — each can flag code the other accepts. That is
normal; pick the one whose results you'll act on.


## 6. Decision guide — when to use which, for what job

**Always:**
* **Ruff** for linting **and** formatting. It replaces `flake8 + isort +
  pyupgrade + black` with one fast tool. There's no real reason to skip it.

**Pick a type checker:**
* Use **pyright** if:
  * you work primarily in **VS Code** (you get it free via Pylance),
  * you want the **fastest** feedback / watch mode,
  * you like stricter inference catching more at the cost of occasional noise.
* Use **mypy** if:
  * you depend on its **plugins** (pydantic, django, sqlalchemy),
  * you want the **reference** checker / maximum ecosystem parity,
  * your team/CI is already standardized on it.

**Common, robust setup — use them together:**
* **Editor:** Pylance/pyright for instant inline feedback.
* **CI:** `ruff check` + `ruff format --check` + **one** type checker
  (mypy *or* pyright — running both in CI is possible but usually redundant).

```yaml
# .github/workflows/ci.yml (excerpt)
- run: pip install ruff mypy
- run: ruff check .
- run: ruff format --check .
- run: mypy src/
```

```toml
# pyproject.toml — ruff + mypy side by side
[tool.ruff.lint]
select = ["E", "F", "I", "UP", "B", "SIM"]

[tool.mypy]
python_version = "3.12"
strict = true
```


## 7. Summary

| Job | Tool |
| --- | ---- |
| Format code (Black-compatible) | **ruff format** |
| Lint: unused imports, bugs, style, import sorting, upgrades | **ruff check** |
| Verify type correctness (static contracts) | **mypy** *or* **pyright** |
| Fast editor type feedback in VS Code | **pyright** (Pylance) |
| Plugin-driven framework typing (pydantic/django) | **mypy** |
| Enforce types **at runtime** | none of these — see [`typed_contracts.ipynb`](typed_contracts.ipynb) |

> **One-liner:** *Ruff lints & formats; mypy/pyright type-check; none of them
> enforce types at runtime.* Use Ruff + one type checker, and add runtime
> validation (pydantic/beartype) at your data boundaries.

### References
* mypy: https://mypy.readthedocs.io/
* pyright: https://microsoft.github.io/pyright/ ·
  Pylance: https://marketplace.visualstudio.com/items?itemName=ms-python.vscode-pylance
* ruff: https://docs.astral.sh/ruff/
* typing spec: https://typing.readthedocs.io/
